## Pruned Fourier-Morse Model $V(D, r_e)$ and $V(D, r_e, \alpha)$

Build and compare anisotropic Morse models: a full Fourier expansion in $(D, re, \alpha)$ with a pruned (sparse-coefficient) version.

---

### Data Exploration & Model Parameter Choice

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from chimorse.config import (
    FigureContext, get_colors, PLOT_PARAMS, MOLECULES
)
from chimorse.dataio import load_data
from chimorse.fitting import generate_fourier_morse_data
from chimorse.plotting import (
    plot_ER, plot_chi_psi_panel,
    plot_prune_coefficients, plot_parity,
    plot_parity_angle_color,
    plot_energy_surfaces_chi_psi
)

plt.rcParams.update(PLOT_PARAMS)

In [ ]:
molecule_name = 'PA'
interaction = 'OP'
zero_zeta = True

alpha_fit = True

harmonic_ceils = {'EP': (8, 1), 'EA': (8, 1), 'OP': (20, 1), 'OA': (20, 1)}

molecule = MOLECULES[molecule_name]

df_org = load_data(molecule, interaction, zero_zeta)
colors = get_colors()

ctx_org = FigureContext(base="Figures", molecule=molecule.name, data_type="org", interaction=interaction)
ctx_model = FigureContext(base="Figures", molecule=molecule.name, data_type="model", interaction=interaction)
ctx_compare = FigureContext(base="Figures", molecule=molecule.name, data_type="compare", interaction=interaction)

### Prune threshold analysis

In [ ]:
relative_th = np.logspace(-5, 0, 30)
plot_prune_coefficients(df_org, molecule, interaction,
                        harmonic_ceils, relative_th, alpha_fit,
                        print_errors=False, colors=colors, save_path=ctx_model.dir())

### Fourier-Morse Model
Fit the anisotropic Morse model with $D(\chi, \psi)$, $r_e(\chi, \psi)$ (and $\alpha(\chi, \psi)$) as full Fourier expansions, generate the model energy surface, and compare it to the reference data.

In [ ]:
prune_model = True
prune_top_n = {'D': 40, 're': 45, 'alpha':80}

df_model = generate_fourier_morse_data(df_org, molecule, interaction, harmonic_ceils, 
                                alpha_fit=alpha_fit, original_size=True, print_errors=True, 
                                prune_model=prune_model, prune_top_n=prune_top_n
                                )

### Visualization Comparison

In [ ]:
parity_plot_mode = ['Emin', 're', 'E'][2]

plot_ER(df_model, show_zoom_box=True, alpha_fit=alpha_fit, save_path=ctx_model.dir())
plot_chi_psi_panel(df_model, interaction,  molecule.screw_step, colors = colors, 
                   left_label=True, save_path=ctx_model.dir())

plot_parity(df_org, df_model, interaction, alpha_fit=alpha_fit, colors=colors, 
            compare_target=parity_plot_mode, save_path=ctx_compare.dir())
plot_parity_angle_color(df_org, df_model, color_by='chi', zoom_in=5, save_path=ctx_model.dir())
plot_energy_surfaces_chi_psi(df_org, interaction, molecule.screw_step, colors, model_df=df_model, 
                             plot_type='difference', left_label=True, save_path=ctx_compare.dir())